# **Transform Drivers Data**
  1. Read bronze Drivers table
  2. Keep only the columns required for analytics (Drop url column)
  3. Standardise Column using snake_case (driverid -> driver_id,dateofbirth -> date_of_birth)
  4. Concatenate name.givenName and name.familyName to create new column called driver_name and tranform value to the Title Case 
  5. Remove duplicate records 
  6. Transform values of columns nationality to Title Case 
  7. Write transformated data to drivers table 

### Entity Relationship Diagram - Forumula1 Bronze Schema
![](/Workspace/Users/uppalapatiususp@gmail.com/Pavan_Azure_DataEngineer_Projects/Formula1_Project/03-silver/EntityRelation_Diagram.png)




In [0]:
%run ../00-common/01_Environment-Config

In [0]:
bronze_table_nm = f"{catalog_name}.{bronze_schema}.drivers"
silver_table_nm = f"{catalog_name}.{silver_schema}.drivers"

### Step 1 - Read bronze Drivers table

In [0]:
# #Below is one way of reading data from table 
# drivers_df = (
#     spark.table(bronze_table_nm)    
# )

In [0]:
#Below is another way of reading data from table this one allows to add options 
drivers_df = (
    spark.read.table(bronze_table_nm)   
    
)

In [0]:
display(drivers_df)

### Step 2 - Keep only the columns requried for analytics (Drop Url column)

In [0]:
from pyspark.sql import functions as F

In [0]:
drivers_dropped_df =  drivers_df.drop("url")
    

### Step 3 - Standardise Column using snake_case (driverid -> driver_id,dateofbirth -> date_of_birth)
Standardise Column using snake_case (constructorid -> constructor_id)

In [0]:
drivers_renamed_df =  drivers_dropped_df.withColumnsRenamed({"driverId": "driver_Id", "dateOfBirth":"date_of_birth"})


In [0]:
display(drivers_renamed_df)

### Step -4 Concatenate name.givenName and name.familyName to create new column called driver_name and tranform value to the Title Case 

In [0]:
 drivers_concat_df = (
        drivers_renamed_df
            .withColumn("driver_name",
                        F.initcap(F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName")))
                        )
            .drop("name")
    
    )

In [0]:
display(drivers_concat_df)

### Step - 5 Remove Duplicate Records

In [0]:
drivers_duplicate_df = drivers_concat_df.dropDuplicates(["driver_Id"])

### Step - 6 Transform values of columns nationality to Title Case 

In [0]:
drivers_final_df = drivers_duplicate_df.withColumn("nationality", F.initcap(F.col("nationality")))

### Step - 7. Write transformated data to drivers table 

In [0]:
(
    drivers_final_df
        .write.mode("overwrite")
        .format("delta")
        .saveAsTable(silver_table_nm)
)

In [0]:
spark.table(silver_table_nm).display()